In [7]:
import anndata as ad
import scanpy as sc
import pandas as pd
import gseapy as gp
from openai import OpenAI

# ============ Step 1. 读取数据 ============
adata = ad.read_h5ad("/data2/xiaoxinyu/project/gene_text_pairs/DLPFC/gene-h5ad/AAACAATCTACTAGCA-1-1-0-0.h5ad")

# 获取组织区域注释
region_label = adata.obs["region_label"].unique()[0]

# 对应的中文描述
region_map = {
    "Layer_1": "大脑皮层的分子层。",
    "Layer_2": "大脑皮层的外颗粒层。",
    "Layer_3": "大脑皮层的外锥体层。",
    "Layer_4": "大脑皮层的内颗粒层。",
    "Layer_5": "大脑皮层的内锥体层。",
    "Layer_6": "大脑皮层的多形层。",
    "WM": "大脑的白质层。"
}
region_desc = region_map.get(region_label, "未知区域")

# ============ Step 2. 获取 Top10 高表达基因 ============
# 取均值最高的基因
gene_exp = adata.to_df().mean(axis=0)
top10_genes = gene_exp.sort_values(ascending=False).head(10).index.tolist()

# ============ Step 3. 单样本富集分析 (ssGSEA) ============
# 用 gseapy 需要输入 gene expression 的 DataFrame，行为基因，列为样本
exp_df = pd.DataFrame(gene_exp, columns=["expression"])
exp_df = exp_df.reset_index()
exp_df.columns = ["Gene", "Expression"]

# 准备 GMT 基因集（这里示例用 KEGG，可以换成 Reactome/MSigDB）
ssgsea_res = gp.ssgsea(
    data=adata.to_df().T,  # 基因 × 样本
    gene_sets='KEGG_2021_Human', # 也可用 "GO_Biological_Process_2021"
    sample_norm_method='rank',
    outdir=None,
    verbose=True
)
# print(ssgsea_res.res2d.head())
top3_pathways = (
    ssgsea_res.res2d.sort_values(by="ES", ascending=False)
      .head(4)["Term"]
      .tolist()
)

# ============ Step 4. 拼接 GPT 输入信息 ============
gpt_input = f"""
组织区域注释：{region_label}（{region_desc}）
前10个高表达基因：{', '.join(top10_genes)}
前3个富集通路：{', '.join(top3_pathways)}
"""

print(gpt_input)

2025-08-31 15:54:18,114 [INFO] Parsing data files for ssGSEA...........................
2025-08-31 15:54:18,135 [INFO] Enrichr library gene sets already downloaded in: /home/xiaoxinyu/.cache/gseapy, use local file


2025-08-31 15:54:18,150 [INFO] 0020 gene_sets have been filtered out when max_size=500 and min_size=15
2025-08-31 15:54:18,150 [INFO] 0300 gene_sets used for further statistical testing.....
2025-08-31 15:54:18,151 [INFO] Start to run ssGSEA...Might take a while................



组织区域注释：Layer_1（大脑皮层的分子层。）
前10个高表达基因：MT-CO3, MT-CO1, MT-ND1, MT-ATP6, MT-CO2, MT-ND4, MT-ND2, MT-CYB, MT-ND3, COX6C
前3个富集通路：Ribosome, Oxidative phosphorylation, Protein export, Proteasome



In [8]:
# ============ Step 5. 调用 GPT API ============

prompt = f"""
你是一位生物医学专家。我将提供一个组织样本的结构化信息，请你生成一段科研风格的简洁描述（适合作为空间转录组研究报告中的样本注释）。

输入信息：
{gpt_input}

要求：
1. 使用科研风格（而不是科普风格）。
2. 输出为一段连贯的文字，而不是简单的条目。
3. 先提及组织区域注释。
4. 融合基因功能：对前10个基因，每个用2–3句话总结其主要功能、典型高表达组织/细胞、以及与该样本的潜在相关性。
5. 融合通路信息：对前3个通路，每个用3–4句话总结其核心生物学作用，以及在该样本背景下的潜在意义。
6. 最终文本应读起来像是对该样本表达谱的生物学解释。
"""

client = OpenAI(
    api_key='sk-YvgrWZx5GUDmQo4y4fE9961316D84aCaB4AcAb08A298D36f',
    base_url="https://yeysai.com/v1",
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # 可以换成更大的模型 gpt-4o
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7
)

final_text = response.choices[0].message.content
print("==== 最终样本文本 ====")
print(final_text)


==== 最终样本文本 ====
该组织样本来源于大脑皮层的分子层（Layer 1），该区域以其丰富的神经元突触联系和神经调节功能著称，具有在信息处理和感知中的重要作用。根据基因表达谱，样本中前十个高表达基因主要涉及线粒体功能和氧化磷酸化过程。

首先，MT-CO3、MT-CO1、MT-ND1、MT-ATP6、MT-CO2、MT-ND4、MT-ND2、MT-CYB和MT-ND3这几种基因都与线粒体的氧化磷酸化链有关，承担电子传递与ATP合成的核心功能。这些基因的高表达在脑组织中尤其突出，表明该区域可能具有较高的代谢需求。MT-CO3和MT-CO1编码线粒体复合体IV的亚基，而MT-ND1、MT-ND2、MT-ND3、MT-ND4分别编码复合体I的不同亚单位，这些基因的高表达是细胞能量供应的基础，特别是在代谢活跃的神经组织中。COX6C则是与线粒体呼吸链复合体的调节相关，进一步支持了该区域的高能需求。

在功能通路方面，样本中显著富集的“Ribosome”通路涉及核糖体的合成与蛋白质翻译过程，这对于维持神经元的细胞功能至关重要。尤其是在大脑皮层区域，蛋白质合成的高需求有助于突触可塑性和神经元之间的信息传递。其次，“Oxidative phosphorylation”通路是能量生成的核心机制，线粒体通过此过程生成ATP，为神经元的电活动和信息处理提供能量支持。大脑皮层尤其依赖这一过程以维持其高度的代谢活动。最后，“Protein export”通路与蛋白质的合成、折叠与运输密切相关，保证了神经元中蛋白质的正常功能及其在突触信号传递中的角色。该通路的高表达提示样本可能在蛋白质的质量控制与功能调节方面具有重要的生物学意义。

综合来看，本样本的表达谱显示出线粒体相关基因和代谢通路的显著活跃，反映出大脑皮层分子层在维持细胞代谢、蛋白质合成与能量供应方面的关键作用。
